In [1]:
########################################################################
# Inclusão das Bibliotecas Necessárias
########################################################################
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
########################################################################
# Localizando o Diretório Base
########################################################################
%cd /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código


/content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código


In [ ]:
# =============================

In [3]:
"""
DIAGNÓSTICO COMPLETO DA GPU CUDA NO GOOGLE COLAB
- Identifica modelo da GPU
- Mostra memória disponível
- Verifica capacidade CUDA
- Testa performance básica
"""

import torch
import numpy as np
import time
import psutil
import platform
import subprocess
import sys
from datetime import datetime

def print_separator(char="=", length=80):
    print(char * length)

def check_cuda_availability():
    """Verifica se CUDA está disponível"""
    print_separator()
    print("🚀 VERIFICANDO CUDA")
    print_separator()

    cuda_available = torch.cuda.is_available()
    print(f"CUDA disponível: {cuda_available}")

    if not cuda_available:
        print("\n⚠️ CUDA NÃO ESTÁ DISPONÍVEL!")
        print("Possíveis causas:")
        print("  1. GPU não habilitada no Colab (Runtime → Change runtime type → GPU)")
        print("  2. Driver não instalado")
        print("  3. PyTorch sem suporte CUDA")
        return False

    print(f"Versão do CUDA: {torch.version.cuda}")
    print(f"PyTorch versão: {torch.__version__}")
    print(f"GPU ativa: {torch.cuda.is_available()}")
    print(f"Número de GPUs: {torch.cuda.device_count()}")

    return True

def get_gpu_info():
    """Obtém informações detalhadas da GPU"""
    print_separator()
    print("🎮 INFORMAÇÕES DA GPU")
    print_separator()

    for i in range(torch.cuda.device_count()):
        print(f"\n--- GPU {i} ---")
        print(f"  Nome: {torch.cuda.get_device_name(i)}")
        print(f"  Capacidade de computação: {torch.cuda.get_device_capability(i)}")

        # Memória total
        total_memory = torch.cuda.get_device_properties(i).total_memory
        print(f"  Memória total: {total_memory / 1024**3:.2f} GB")

        # Memória livre atual
        free_memory = torch.cuda.memory_reserved(i) - torch.cuda.memory_allocated(i)
        print(f"  Memória livre (reservada): {free_memory / 1024**2:.2f} MB")

        # Memória alocada
        allocated = torch.cuda.memory_allocated(i)
        print(f"  Memória alocada: {allocated / 1024**2:.2f} MB")

        # Memória máxima já alocada
        max_allocated = torch.cuda.max_memory_allocated(i)
        print(f"  Pico de memória alocada: {max_allocated / 1024**2:.2f} MB")

        print(f"  Multiprocessadores: {torch.cuda.get_device_properties(i).multi_processor_count}")
        print(f"  Clock rate: {torch.cuda.get_device_properties(i).clock_rate / 1000:.0f} MHz")

def get_gpu_details_nvidia_smi():
    """Obtém detalhes usando nvidia-smi (mais completo)"""
    print_separator()
    print("🔧 DETALHES NVIDIA-SMI")
    print_separator()

    try:
        # Executa nvidia-smi
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu,temperature.gpu,power.draw',
                                '--format=csv,noheader,nounits'],
                                capture_output=True, text=True)

        if result.returncode == 0:
            lines = result.stdout.strip().split('\n')
            for i, line in enumerate(lines):
                if line:
                    parts = [x.strip() for x in line.split(',')]
                    print(f"\n--- GPU {i} (nvidia-smi) ---")
                    if len(parts) >= 7:
                        print(f"  Nome: {parts[0]}")
                        print(f"  Memória Total: {parts[1]} MB ({float(parts[1])/1024:.2f} GB)")
                        print(f"  Memória Usada: {parts[2]} MB ({float(parts[2])/1024:.2f} GB)")
                        print(f"  Memória Livre: {parts[3]} MB ({float(parts[3])/1024:.2f} GB)")
                        print(f"  Utilização GPU: {parts[4]}%")
                        print(f"  Temperatura: {parts[5]}°C")
                        print(f"  Consumo de energia: {parts[6]} W")
        else:
            print("⚠️ Não foi possível executar nvidia-smi")
    except Exception as e:
        print(f"⚠️ Erro ao executar nvidia-smi: {e}")

def test_cuda_performance():
    """Testa a performance da GPU CUDA"""
    print_separator()
    print("⚡ TESTE DE PERFORMANCE")
    print_separator()

    # Configurações do teste
    sizes = [100, 500, 1000, 2000, 5000]

    print("\nComparação CPU vs GPU para multiplicação de matrizes:")
    print("-" * 60)
    print(f"{'Tamanho':<12} {'CPU (s)':<12} {'GPU (s)':<12} {'Aceleração':<12}")
    print("-" * 60)

    for size in sizes:
        # Matrizes aleatórias
        a = np.random.randn(size, size).astype(np.float32)
        b = np.random.randn(size, size).astype(np.float32)

        # CPU
        start_cpu = time.time()
        c_cpu = np.dot(a, b)
        cpu_time = time.time() - start_cpu

        # GPU
        a_gpu = torch.tensor(a).cuda()
        b_gpu = torch.tensor(b).cuda()

        # Aquecer a GPU
        _ = torch.mm(a_gpu, b_gpu)
        torch.cuda.synchronize()

        start_gpu = time.time()
        c_gpu = torch.mm(a_gpu, b_gpu)
        torch.cuda.synchronize()
        gpu_time = time.time() - start_gpu

        # Aceleração
        speedup = cpu_time / gpu_time if gpu_time > 0 else 0

        print(f"{size:<12} {cpu_time:<12.4f} {gpu_time:<12.4f} {speedup:<12.2f}x")

    print("-" * 60)

def test_memory_bandwidth():
    """Testa a largura de banda da memória GPU"""
    print_separator()
    print("💾 TESTE DE LARGURA DE BANDA DE MEMÓRIA")
    print_separator()

    sizes = [100, 500, 1000, 2000]

    print("\nTransferência CPU → GPU → CPU:")
    print("-" * 60)
    print(f"{'Tamanho (MB)':<15} {'Transferência (ms)':<20} {'Largura de banda (GB/s)':<25}")
    print("-" * 60)

    for size in sizes:
        # Tamanho em MB
        mb = size * size * 4 / 1024 / 1024  # float32 = 4 bytes

        # CPU → GPU
        cpu_tensor = torch.randn(size, size, dtype=torch.float32)
        start = time.time()
        gpu_tensor = cpu_tensor.cuda()
        torch.cuda.synchronize()
        transfer_time = (time.time() - start) * 1000  # ms

        # GPU → CPU
        start = time.time()
        cpu_back = gpu_tensor.cpu()
        torch.cuda.synchronize()
        transfer_time += (time.time() - start) * 1000

        # Largura de banda (dados totais = ida + volta = 2 * mb)
        bandwidth = (2 * mb) / (transfer_time / 1000)  # GB/s

        print(f"{mb:.1f} MB{' ':<10} {transfer_time:.2f} ms{' ':<10} {bandwidth:.2f} GB/s")

    print("-" * 60)

def test_memory_allocation():
    """Testa alocação máxima de memória GPU"""
    print_separator()
    print("🧠 TESTE DE ALOCAÇÃO DE MEMÓRIA")
    print_separator()

    device = torch.cuda.current_device()
    total_memory = torch.cuda.get_device_properties(device).total_memory
    print(f"Memória total: {total_memory / 1024**3:.2f} GB")

    # Tentar alocar diferentes tamanhos
    sizes_mb = [100, 500, 1000, 2000, 3000, 4000, 5000]

    print("\nTestando alocação de tensores:")
    print("-" * 50)

    for size_mb in sizes_mb:
        try:
            # Calcular tamanho do tensor (assumindo float32 = 4 bytes)
            num_elements = int(size_mb * 1024 * 1024 / 4)
            size_side = int(np.sqrt(num_elements))

            # Tentar alocar
            tensor = torch.randn(size_side, size_side, dtype=torch.float32).cuda()
            allocated = tensor.element_size() * tensor.nelement()
            print(f"  ✅ {size_mb:4d} MB → tensor {size_side}x{size_side} → {allocated / 1024**2:.1f} MB alocados")

            # Liberar
            del tensor
            torch.cuda.empty_cache()
        except RuntimeError as e:
            print(f"  ❌ {size_mb:4d} MB → Falha: {str(e)[:50]}...")
            break

    torch.cuda.empty_cache()

def get_system_info():
    """Obtém informações do sistema"""
    print ("########################################################################")
    print ("# INFORMAÇÕES DO SISTEMA")
    print ("########################################################################")
    print(f"# Sistema operacional .....: {platform.system()} {platform.release()}")
    print(f"# Arquitetura .............: {platform.machine()}")
    print(f"# Processador .............: {platform.processor()}")
    print(f"# Python versão ...........: {sys.version}")
    print(f"# RAM total ...............: {psutil.virtual_memory().total / 1024**3:.2f} GB")
    print(f"# RAM disponível ..........: {psutil.virtual_memory().available / 1024**3:.2f} GB")
    print(f"# RAM usada ...............: {psutil.virtual_memory().used / 1024**3:.2f} GB")
    print(f"# CPU cores ...............: {psutil.cpu_count()}")
    print(f"# CPU frequência ..........: {psutil.cpu_freq().current:.0f} MHz" if psutil.cpu_freq() else "CPU frequência: N/A")
    print ("########################################################################")

def check_cuda_capabilities():
    """Verifica capacidades específicas da CUDA"""
    print_separator()
    print("🔍 CAPACIDADES CUDA")
    print_separator()

    if not torch.cuda.is_available():
        print("CUDA não disponível")
        return

    capabilities = {
        'Tensor Cores': torch.cuda.has_half,
        'Float16 suporte': torch.cuda.is_bf16_supported(),
        'FP16 (half)': True,
        'FP32 (float)': True,
        'FP64 (double)': torch.cuda.is_bf16_supported(),
    }

    print("\nCapacidades de precisão:")
    for cap, value in capabilities.items():
        print(f"  {cap}: {'✅ Sim' if value else '❌ Não'}")

    # Verificar suporte a mixed precision
    try:
        from torch.cuda.amp import autocast
        print("  Mixed Precision (AMP): ✅ Suportado")
    except:
        print("  Mixed Precision (AMP): ❌ Não suportado")

def get_current_gpu_status():
    """Obtém status atual da GPU"""
    print_separator()
    print("📊 STATUS ATUAL DA GPU")
    print_separator()

    if not torch.cuda.is_available():
        print("CUDA não disponível")
        return

    device = torch.cuda.current_device()

    print(f"\nDispositivo atual: {torch.cuda.current_device()}")
    print(f"GPU atual: {torch.cuda.get_device_name(device)}")

    # Memória
    allocated = torch.cuda.memory_allocated(device)
    reserved = torch.cuda.memory_reserved(device)
    free = reserved - allocated
    total = torch.cuda.get_device_properties(device).total_memory

    print ("########################################################################")
    print(f"# Memória ")
    print ("########################################################################")
    print(f"# Total ................: {total / 1024**3:.2f} GB")
    print(f"# Reservada ............: {reserved / 1024**2:.2f} MB")
    print(f"# Alocada ..............: {allocated / 1024**2:.2f} MB")
    print(f"# Livre (reservada) ....: {free / 1024**2:.2f} MB")
    print(f"# Utilização ...........: {allocated / total * 100:.1f}%")
    print ("########################################################################")

    # Cache
    print(f"\nCache:")
    print(f"  Memória cache: {torch.cuda.memory_cached(device) / 1024**2:.2f} MB")

def simple_training_test():
    """Teste simples de treinamento para verificar funcionamento"""
    print_separator()
    print("🧪 TESTE SIMPLES DE TREINAMENTO")
    print_separator()

    print("\nCriando modelo simples...")

    # Modelo simples
    class SimpleModel(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = torch.nn.Linear(784, 256)
            self.fc2 = torch.nn.Linear(256, 128)
            self.fc3 = torch.nn.Linear(128, 10)

        def forward(self, x):
            x = torch.relu(self.fc1(x))
            x = torch.relu(self.fc2(x))
            return self.fc3(x)

    model = SimpleModel().cuda()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = torch.nn.CrossEntropyLoss()

    # Dados sintéticos
    batch_size = 64
    input_data = torch.randn(batch_size, 784).cuda()
    labels = torch.randint(0, 10, (batch_size,)).cuda()

    # Forward pass
    start = time.time()
    outputs = model(input_data)
    loss = criterion(outputs, labels)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    torch.cuda.synchronize()
    elapsed = (time.time() - start) * 1000

    print(f"✅ Forward + Backward completado em {elapsed:.2f} ms")
    print(f"Loss: {loss.item():.4f}")

    # Limpeza
    del model, optimizer, input_data, labels
    torch.cuda.empty_cache()

def main():
    """Função principal"""
    print ("# ================================================================" )
    print("🔬 DIAGNÓSTICO COMPLETO DA GPU CUDA - GOOGLE COLAB")
    print(f"Data: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print_separator("=", 80)

    # 1. Verificar CUDA
    if not check_cuda_availability():
        print("\n❌ Por favor, habilite a GPU no Colab:")
        print("   Runtime → Change runtime type → GPU → Save")
        return

    # 2. Informações básicas da GPU
    get_gpu_info()

    # 3. Detalhes do nvidia-smi
    get_gpu_details_nvidia_smi()

    # 4. Informações do sistema
    get_system_info()

    # 5. Capacidades CUDA
    check_cuda_capabilities()

    # 6. Status atual da GPU
    get_current_gpu_status()

    # 7. Teste de performance
    test_cuda_performance()

    # 8. Teste de largura de banda
    test_memory_bandwidth()

    # 9. Teste de alocação de memória
    test_memory_allocation()

    # 10. Teste simples de treinamento
    simple_training_test()

    # Resumo final
    print_separator()
    print("📋 RESUMO FINAL")
    print_separator()

    device = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(device)
    total_memory = torch.cuda.get_device_properties(device).total_memory / 1024**3
    cuda_version = torch.version.cuda

    print(f"\n✅ GPU Configuração:")
    print(f"   • Modelo: {gpu_name}")
    print(f"   • Memória: {total_memory:.2f} GB")
    print(f"   • CUDA: {cuda_version}")
    print(f"   • PyTorch: {torch.__version__}")

    # Recomendações
    print(f"\n💡 RECOMENDAÇÕES:")
    if "T4" in gpu_name:
        print("   • GPU Tesla T4 - Boa para treinamento médio")
        print("   • Use batch size entre 128-256")
        print("   • Até 2-3 camadas de redes neurais")
        print("   • Aproveite os Tensor Cores (use float16)")
    elif "V100" in gpu_name:
        print("   • GPU Tesla V100 - Excelente performance!")
        print("   • Batch size até 512")
        print("   • Ideal para modelos grandes")
        print("   • Use mixed precision para melhor performance")
    elif "P100" in gpu_name:
        print("   • GPU Tesla P100 - Boa performance")
        print("   • Batch size até 256")
        print("   • Sem Tensor Cores, use float32")
    elif "K80" in gpu_name:
        print("   • GPU Tesla K80 - Performance limitada")
        print("   • Use batch size pequeno (64-128)")
        print("   • Evite modelos muito grandes")

    print_separator()
    print("✅ DIAGNÓSTICO CONCLUÍDO!")
    print_separator()

if __name__ == "__main__":
    main()

# ================================================================
🔬 DIAGNÓSTICO COMPLETO DA GPU CUDA - GOOGLE COLAB
Data: 2026-06-12 11:46:35
🚀 VERIFICANDO CUDA
CUDA disponível: True
Versão do CUDA: 12.8
PyTorch versão: 2.11.0+cu128
GPU ativa: True
Número de GPUs: 1
🎮 INFORMAÇÕES DA GPU

--- GPU 0 ---
  Nome: Tesla T4
  Capacidade de computação: (7, 5)
  Memória total: 14.56 GB
  Memória livre (reservada): 0.00 MB
  Memória alocada: 0.00 MB
  Pico de memória alocada: 0.00 MB
  Multiprocessadores: 40
  Clock rate: 1590 MHz
🔧 DETALHES NVIDIA-SMI

--- GPU 0 (nvidia-smi) ---
  Nome: Tesla T4
  Memória Total: 15360 MB (15.00 GB)
  Memória Usada: 3 MB (0.00 GB)
  Memória Livre: 14910 MB (14.56 GB)
  Utilização GPU: 0%
  Temperatura: 51°C
  Consumo de energia: 12.64 W
########################################################################
# INFORMAÇÕES DO SISTEMA
########################################################################
# Sistema operacional .....: Linux 6.6.122+
# Arquitetur

/tmp/ipykernel_2050/245804358.py:301: FutureWarning: `torch.cuda.memory_cached` has been renamed to `torch.cuda.memory_reserved`
  print(f"  Memória cache: {torch.cuda.memory_cached(device) / 1024**2:.2f} MB")


500          0.0041       0.0003       11.84       x
1000         0.0054       0.0011       4.78        x
2000         0.0372       0.0061       6.04        x
5000         0.9191       0.0781       11.77       x
------------------------------------------------------------
💾 TESTE DE LARGURA DE BANDA DE MEMÓRIA

Transferência CPU → GPU → CPU:
------------------------------------------------------------
Tamanho (MB)    Transferência (ms)   Largura de banda (GB/s)  
------------------------------------------------------------
0.0 MB           0.35 ms           215.20 GB/s
1.0 MB           0.79 ms           2420.57 GB/s
3.8 MB           2.01 ms           3803.19 GB/s
15.3 MB           7.06 ms           4321.40 GB/s
------------------------------------------------------------
🧠 TESTE DE ALOCAÇÃO DE MEMÓRIA
Memória total: 14.56 GB

Testando alocação de tensores:
--------------------------------------------------
  ✅  100 MB → tensor 5120x5120 → 100.0 MB alocados
  ✅  500 MB → tensor 11448x11

In [ ]:
# =============================